# Lab 5b — Retrieval-Augmented Generation (RAG)

In the previous notebook we fine-tuned Qwen2.5-0.5B on science Q&A. The model got better at answering questions, but it's still limited to whatever it memorized during training. Ask it something outside that scope and it'll either guess or hallucinate.

RAG fixes this by giving the model access to an external knowledge base. Before generating an answer, the system searches for relevant passages and includes them in the prompt. The model reads the passages and uses them to produce a grounded response.

We'll use **OpenSearch** as the search engine — this is the same technology behind many production search systems. By the end of this notebook, we'll compare all three configurations: vanilla, fine-tuned, and fine-tuned with RAG.

## Background: What is RAG and Why Do We Need It?

Language models have a fundamental limitation: they can only work with what's in their weights. Everything a model "knows" was baked in during training, which means:
- Knowledge has a **cutoff date** — anything that happened after training doesn't exist for the model
- The model can **hallucinate** — confidently generate information that sounds plausible but is wrong
- There's no way to **update** its knowledge without retraining or fine-tuning

**Retrieval-Augmented Generation (RAG)** ([Lewis et al. 2020](https://arxiv.org/abs/2005.11401)) addresses this by adding an external knowledge retrieval step before generation. Instead of relying solely on memorized information, the model gets to read relevant documents at query time.

A RAG system has three main components:

**1. Embedding model** — converts text (both documents and queries) into numerical vectors that capture semantic meaning. Similar texts end up close together in vector space. We use `all-MiniLM-L6-v2`, a small (22M parameter) model designed specifically for this.

**2. Vector store / search engine** — stores document embeddings and retrieves the most similar ones given a query vector. We use OpenSearch, which supports both vector similarity search (knn) and traditional keyword search (BM25). Other options include Pinecone, Qdrant, pgvector, Milvus, and ChromaDB.

**3. Generator model** — reads the retrieved documents plus the original question and produces an answer. This is our fine-tuned Qwen model. The retrieved documents get inserted into the prompt as context.

The key libraries for this notebook:

| Library | Role |
|---------|------|
| **sentence-transformers** | Runs the embedding model. Provides a simple `encode()` interface that turns text into vectors. Built on top of transformers. |
| **opensearch-py** | Python client for OpenSearch. Handles index creation, document ingestion, and search queries — both vector (knn) and keyword (BM25). |
| **peft** | Loads the LoRA adapter we trained in lab5a and attaches it to the base model. |

## Prerequisites

Before running this notebook:
1. You must have completed **lab5a** (the LoRA adapter should be saved in `./models/sciq_lora_adapter/`)
2. OpenSearch must be running: `docker-compose up -d` from the Week5 directory

## RAG Flow
![RAG Pipeline](rag_flow.png)

## 1. Setup and imports

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from opensearchpy import OpenSearch
import json
import time

## 2. Verify OpenSearch is running

If this cell fails, make sure you ran `docker-compose up -d` from the terminal.

In [2]:
os_client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9202}],
    http_compress=True,
    use_ssl=False,
    verify_certs=False,
)

info = os_client.info()
print(f"Connected to OpenSearch {info['version']['number']}")
print(f"Cluster: {info['cluster_name']}")

Connected to OpenSearch 2.17.0
Cluster: docker-cluster


## 3. Load models

We need three things:
- The **base Qwen model** (for vanilla comparison)
- The **LoRA adapter** from lab5a (for fine-tuned comparison)
- The **embedding model** (all-MiniLM-L6-v2) for converting text to vectors

In [3]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"
ADAPTER_PATH = "./models/sciq_lora_adapter"

# Pick the best available device: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model (for vanilla comparison)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
).to(device)
base_model.config.pad_token_id = tokenizer.eos_token_id

# Load the fine-tuned version by attaching the LoRA adapter
finetuned_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
).to(device)
finetuned_model.config.pad_token_id = tokenizer.eos_token_id
finetuned_model = PeftModel.from_pretrained(finetuned_model, ADAPTER_PATH)

print("Base model and fine-tuned model loaded.")

Using device: mps


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model and fine-tuned model loaded.


In [4]:
# The embedding model — completely separate from the generator.
# This model turns text into 384-dimensional vectors.
# It doesn't generate text at all; it just measures how similar two pieces of text are.
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Quick test
test_vec = embedder.encode("This is a test sentence")
print(f"Embedding model loaded. Vector dimension: {len(test_vec)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded. Vector dimension: 384


## 4. Prepare the knowledge base

The support passages from SciQ become our knowledge base. Each passage is a short paragraph that explains a scientific concept. We'll embed all of them and store them in OpenSearch so we can search them later.

In [5]:
dataset = load_dataset("allenai/sciq", trust_remote_code=True)

# Collect all support passages that aren't empty
passages = []
for split in ["train", "validation", "test"]:
    for sample in dataset[split]:
        text = sample["support"].strip()
        if len(text) > 50:  # skip very short or empty passages
            passages.append(text)

# Remove duplicates (some passages appear more than once)
passages = list(set(passages))
print(f"Total unique passages for the knowledge base: {len(passages)}")
print(f"\nExample passage:\n{passages[0][:300]}...")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/sciq' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Total unique passages for the knowledge base: 12103

Example passage:
Harmless background radiation comes from radioactive elements in rocks and from cosmic rays. Other sources of radiation, such as radon, are harmful. They may cause illness in living things and damage materials such as metals. Radiation has several uses, including detecting and treating cancer....


### How embeddings work

An embedding model maps text into a fixed-size vector (in our case, 384 dimensions). The key property: texts with similar meaning end up close together in this vector space, even if they use different words.

For example:
- "Plants use sunlight to make food" and "Photosynthesis converts light energy into chemical energy" would have high cosine similarity
- "Plants use sunlight to make food" and "The stock market crashed in 2008" would have low cosine similarity

**Cosine similarity** measures the angle between two vectors — 1.0 means identical direction (very similar), 0.0 means perpendicular (unrelated). This is what we use to find relevant passages: embed the question, embed all passages, find the passages whose vectors point in the most similar direction to the question vector.

The embedding model (`all-MiniLM-L6-v2`) was specifically trained for this task using contrastive learning — it saw millions of pairs of similar and dissimilar texts and learned to push similar pairs together and dissimilar pairs apart in vector space.

## 5. Embed the passages

We run every passage through the embedding model to get its vector representation. This is a one-time cost — once the vectors are in OpenSearch, we only need to embed the *query* at search time.

In [6]:
print(f"Embedding {len(passages)} passages...")

start = time.time()
passage_embeddings = embedder.encode(passages, show_progress_bar=True, batch_size=64)
elapsed = time.time() - start

print(f"Done in {elapsed:.1f}s")
print(f"Embedding shape: {passage_embeddings.shape}")

Embedding 12103 passages...


Batches:   0%|          | 0/190 [00:00<?, ?it/s]

Done in 40.4s
Embedding shape: (12103, 384)


### Vector search and HNSW

Once we have embeddings, we need a way to efficiently find the closest vectors to a query. A brute-force approach would compare the query against every single passage — this works for our 12K passages but would be unacceptably slow for millions of documents.

**HNSW (Hierarchical Navigable Small World)** is the algorithm OpenSearch uses for approximate nearest-neighbor search. It builds a multi-layered graph where each node is a document vector connected to its neighbors. Searching starts at the top layer (sparse, long-range connections) and narrows down through lower layers (dense, short-range connections) — like zooming into a map. It finds the top-k most similar vectors in milliseconds even with millions of documents.

The trade-off is that HNSW is approximate — it might miss the absolute closest vector, but in practice it finds results that are close enough. The speed gain over exact search is typically 100-1000×.

## 6. Create the OpenSearch index

An OpenSearch index is like a database table. We define it with two fields:
- **text**: the passage content (stored as a regular text field, also searchable with keyword/BM25 search)
- **embedding**: the vector representation (used for similarity search)

The `knn_vector` type tells OpenSearch this field contains vectors and should be indexed for fast nearest-neighbor lookup. We're using HNSW (Hierarchical Navigable Small World) which is the standard algorithm for approximate vector search.

In [7]:
INDEX_NAME = "sciq-passages"

# Delete the index if it already exists (for re-runs)
if os_client.indices.exists(index=INDEX_NAME):
    os_client.indices.delete(index=INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'")

# Define the index mapping
index_body = {
    "settings": {
        "index": {
            "knn": True                     # enable vector search on this index
        }
    },
    "mappings": {
        "properties": {
            "text": {
                "type": "text"              # standard full-text field
            },
            "embedding": {
                "type": "knn_vector",       # vector field
                "dimension": 384,           # must match the embedding model output
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "nmslib"
                }
            }
        }
    }
}

os_client.indices.create(index=INDEX_NAME, body=index_body)
print(f"Created index '{INDEX_NAME}' with text + vector fields")

Deleted existing index 'sciq-passages'
Created index 'sciq-passages' with text + vector fields


## 7. Index the passages

Now we push every passage (text + embedding) into OpenSearch. We use the bulk API for speed — sending them one by one would be much slower.

In [8]:
from opensearchpy.helpers import bulk

def generate_bulk_actions(passages, embeddings, index_name):
    """Yield documents formatted for the OpenSearch bulk API."""
    for i, (text, emb) in enumerate(zip(passages, embeddings)):
        yield {
            "_index": index_name,
            "_id": i,
            "_source": {
                "text": text,
                "embedding": emb.tolist()
            }
        }

print(f"Indexing {len(passages)} passages into OpenSearch...")

start = time.time()
success_count, errors = bulk(
    os_client,
    generate_bulk_actions(passages, passage_embeddings, INDEX_NAME),
    chunk_size=200,
    request_timeout=120
)
elapsed = time.time() - start

print(f"Indexed {success_count} documents in {elapsed:.1f}s")
if errors:
    print(f"Errors: {errors}")

# Force a refresh so the documents are immediately searchable
os_client.indices.refresh(index=INDEX_NAME)

Indexing 12103 passages into OpenSearch...
Indexed 12103 documents in 20.5s


{'_shards': {'total': 2, 'successful': 1, 'failed': 0}}

## 8. Search function

This is the retrieval step of RAG. Given a question:
1. Embed it using the same embedding model
2. Search OpenSearch for the k most similar passages (by cosine similarity)
3. Return the matching passages

The embedding model maps questions and passages into the same vector space, so a question about photosynthesis will naturally be close to passages about photosynthesis.

In [9]:
def search_passages(query, top_k=3):
    """Search OpenSearch for passages relevant to the query."""
    query_embedding = embedder.encode(query).tolist()
    
    search_body = {
        "size": top_k,
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": top_k
                }
            }
        }
    }
    
    response = os_client.search(index=INDEX_NAME, body=search_body)
    
    results = []
    for hit in response["hits"]["hits"]:
        results.append({
            "text": hit["_source"]["text"],
            "score": hit["_score"]
        })
    
    return results

In [23]:
# Try a search to see if it works
test_results = search_passages("What is photosynthesis?")

print("Search results for: 'What is photosynthesis?'\n")
for i, r in enumerate(test_results, 1):
    print(f"Result {i} (score: {r['score']:.4f}):")
    print(f"  {r['text'][:200]}...")
    print()

Search results for: 'What is photosynthesis?'

Result 1 (score: 0.8163):
  Photosynthesis , the process of turning the energy of sunlight into ‘‘food,’’ is divided into two basic sets of reactions, known as the light reactions and the Calvin cycle, which uses carbon dioxide....

Result 2 (score: 0.8086):
  Photosynthesis is the process by which plants use energy from sunlight to synthesize carbohydrates....

Result 3 (score: 0.8052):
  The products of photosynthesis are glucose and oxygen....



## 9. Build the RAG pipeline

The full RAG flow:
1. Take the user's question
2. Search for relevant passages
3. Build a prompt that includes the retrieved passages as context
4. Feed it to the fine-tuned model

The prompt template matters. We tell the model explicitly to use the provided context to answer the question. This is sometimes called "prompt augmentation" — we're augmenting the original question with retrieved knowledge.

In [24]:
RAG_PROMPT_TEMPLATE = """Use the following context to answer the question. If the context doesn't contain enough information, say so.

Context:
{context}

Question: {question}
Answer:"""

def generate_answer(model, tokenizer, prompt, max_new_tokens=100):
    """Generate text from a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def answer_with_rag(question, model, tokenizer, top_k=3):
    """Full RAG pipeline: retrieve, augment, generate."""
    # Step 1: Retrieve
    results = search_passages(question, top_k=top_k)
    
    # Step 2: Build context from retrieved passages
    context = "\n\n".join([r["text"] for r in results])
    
    # Step 3: Augment the prompt
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
    
    # Step 4: Generate
    answer = generate_answer(model, tokenizer, prompt)
    
    return answer, results


## 10. Test RAG on a single question

Before running the full comparison, let's walk through one example to see the whole pipeline in action.

In [25]:
test_question = "What is the name of the process by which plants make their own food using sunlight?"

# Vanilla answer (no fine-tuning, no RAG)
vanilla_prompt = f"Question: {test_question}\nAnswer:"
vanilla_ans = generate_answer(base_model, tokenizer, vanilla_prompt)
print("VANILLA:", vanilla_ans)
print()

# Fine-tuned answer (no RAG)
ft_prompt = f"Question: {test_question}\nAnswer:"
ft_ans = generate_answer(finetuned_model, tokenizer, ft_prompt)
print("FINE-TUNED:", ft_ans)
print()

# Fine-tuned + RAG answer
rag_ans, retrieved = answer_with_rag(test_question, finetuned_model, tokenizer)
print("FINE-TUNED + RAG:", rag_ans)
print()
print("--- Retrieved passages ---")
for i, r in enumerate(retrieved, 1):
    print(f"\n[{i}] (score: {r['score']:.4f})")
    print(f"    {r['text'][:150]}...")

VANILLA: The process is called photosynthesis.
 A single-select problem: Is the question answered in a satisfactory fashion?

Available options:
 (1). yes
 (2). no
(1).

FINE-TUNED: photosynthesis

FINE-TUNED + RAG: photosynthesis

--- Retrieved passages ---

[1] (score: 0.7947)
    By far the most common producers use the energy in sunlight to make food. This is called photosynthesis . Producers that photosynthesize include plant...

[2] (score: 0.7675)
    Photosynthesis is the process by which plants use energy from sunlight to synthesize carbohydrates....

[3] (score: 0.7614)
    There are also bacteria that use chemical processes to produce food. They get their energy from sources other than the sun, but they are still called ...


## 11. Full three-way comparison

Now let's run all three configurations on the same 10 test questions from lab5a.

In [26]:
TEST_QUESTIONS = [
    "What type of organism is commonly used in preparation of foods such as bread and yogurt?",
    "What is the term for the property of a mineral that describes how it reflects light?",
    "What phenomenon makes global sea levels rise when ice sheets melt?",
    "What is the main gas found in the air we breathe?",
    "What is the name of the process by which plants make their own food using sunlight?",
    "What are the building blocks of proteins called?",
    "What type of rock is formed from cooled magma or lava?",
    "What force keeps planets in orbit around the sun?",
    "What is the smallest unit of life that can replicate independently?",
    "What part of the cell contains the genetic material?"
]

EXPECTED_ANSWERS = [
    "Microorganisms (yeast, bacteria)",
    "Luster",
    "Thermal expansion of water",
    "Nitrogen",
    "Photosynthesis",
    "Amino acids",
    "Igneous rock",
    "Gravity",
    "Cell",
    "Nucleus"
]

In [27]:
vanilla_answers = []
finetuned_answers = []
rag_answers = []

for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"Processing question {i}/{len(TEST_QUESTIONS)}...", end="\r")
    
    # Vanilla
    prompt = f"Question: {q}\nAnswer:"
    vanilla_answers.append(generate_answer(base_model, tokenizer, prompt))
    
    # Fine-tuned (no RAG)
    finetuned_answers.append(generate_answer(finetuned_model, tokenizer, prompt))
    
    # Fine-tuned + RAG
    rag_ans, _ = answer_with_rag(q, finetuned_model, tokenizer)
    rag_answers.append(rag_ans)

print("Done!                          ")

Done!                          


In [28]:
# Print the full comparison
for i, q in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*80}")
    print(f"Q{i+1}: {q}")
    print(f"Expected: {EXPECTED_ANSWERS[i]}")
    print(f"-"*80)
    print(f"  Vanilla:        {vanilla_answers[i]}")
    print(f"  Fine-tuned:     {finetuned_answers[i]}")
    print(f"  Fine-tuned+RAG: {rag_answers[i]}")


Q1: What type of organism is commonly used in preparation of foods such as bread and yogurt?
Expected: Microorganisms (yeast, bacteria)
--------------------------------------------------------------------------------
  Vanilla:        The most common type of organism used in the production of bread and yogurt is the yeast.
 A single-select problem: Is the question answered in a satisfactory fashion?

Pick from:
 (1). yes
 (2). no
(1).
  Fine-tuned:     bacteria
  Fine-tuned+RAG: bacteria

Q2: What is the term for the property of a mineral that describes how it reflects light?
Expected: Luster
--------------------------------------------------------------------------------
  Vanilla:        The term for the property of a mineral that describes how it reflects light is called refractive index.
 A single-select problem: Is the question answered in a satisfactory fashion?

Choose from:
 (1). yes
 (2). no
(1).
  Fine-tuned:     optical property
  Fine-tuned+RAG: luster

Q3: What phenomenon

## 12. Results summary

Let's make a cleaner table and count how many questions each configuration gets right (or close to right).

In [16]:
for i in range(len(TEST_QUESTIONS)):
    print(f"{'='*70}")
    print(f"Q{i+1}: {TEST_QUESTIONS[i]}")
    print(f"  Expected:        {EXPECTED_ANSWERS[i]}")
    print(f"  Vanilla:         {vanilla_answers[i]}")
    print(f"  Fine-tuned:      {finetuned_answers[i]}")
    print(f"  Fine-tuned+RAG:  {rag_answers[i]}")
    print()

Q1: What type of organism is commonly used in preparation of foods such as bread and yogurt?
  Expected:        Microorganisms (yeast, bacteria)
  Vanilla:         The most common type of organism used in the production of bread and yogurt is the yeast.
 A single-select problem: Is the question answered in a satisfactory fashion?

Pick from:
 (1). yes
 (2). no
(1).
  Fine-tuned:      bacteria
  Fine-tuned+RAG:  bacteria

Q2: What is the term for the property of a mineral that describes how it reflects light?
  Expected:        Luster
  Vanilla:         The term for the property of a mineral that describes how it reflects light is called refractive index.
 A single-select problem: Is the question answered in a satisfactory fashion?

Choose from:
 (1). yes
 (2). no
(1).
  Fine-tuned:      optical property
  Fine-tuned+RAG:  luster

Q3: What phenomenon makes global sea levels rise when ice sheets melt?
  Expected:        Thermal expansion of water
  Vanilla:         The melting of ice she

## 13. Inspecting the retrieval step

One of the advantages of RAG over pure generation is transparency — you can always see *what* the model was given to work with. Let's look at the retrieved passages for a few questions.

In [29]:
# Pick a few interesting questions to inspect
inspect_indices = [0, 4, 6]  # microorganisms, photosynthesis, igneous rock

for idx in inspect_indices:
    q = TEST_QUESTIONS[idx]
    results = search_passages(q, top_k=3)
    
    print(f"\n{'='*80}")
    print(f"Question: {q}")
    print(f"RAG answer: {rag_answers[idx]}")
    print(f"\nRetrieved passages:")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] score={r['score']:.4f}")
        print(f"      {r['text'][:200]}")
        print()


Question: What type of organism is commonly used in preparation of foods such as bread and yogurt?
RAG answer: bacteria

Retrieved passages:
  [1] score=0.6998
      Yogurt is a good source of calcium. Yogurt also contains active cultures of "good" bacteria. Foods that contain these beneficial bacteria are sometimes called "probiotic. ".

  [2] score=0.6956
      Humans have collected and grown mushrooms for food for thousands of years. Figure below shows some of the many types of mushrooms that people eat. Yeasts are used in bread baking and brewing alcoholic

  [3] score=0.6901
      Bacteria can be used to make cheese from milk. The bacteria turn the milk sugars into lactic acid. The acid is what causes the milk to curdle to form cheese. Bacteria are also involved in producing ot


Question: What is the name of the process by which plants make their own food using sunlight?
RAG answer: photosynthesis

Retrieved passages:
  [1] score=0.7947
      By far the most common producers use

## 14. Try your own questions

The cell below lets you ask any science question and see all three answers side by side. Try questions that are clearly outside the training data to see where RAG really shines.

In [30]:
# Change this to whatever you want to ask
your_question = "What is a nuclear reactor?"

# Vanilla
prompt = f"Question: {your_question}\nAnswer:"
v = generate_answer(base_model, tokenizer, prompt)

# Fine-tuned
ft = generate_answer(finetuned_model, tokenizer, prompt)

# RAG
r, retrieved = answer_with_rag(your_question, finetuned_model, tokenizer)

print(f"Question: {your_question}\n")
print(f"Vanilla:        {v}")
print(f"Fine-tuned:     {ft}")
print(f"Fine-tuned+RAG: {r}")
print(f"\nRetrieved context:")
for i, p in enumerate(retrieved, 1):
    print(f"  [{i}] {p['text'][:150]}...")

Question: What is a nuclear reactor?

Vanilla:        The first nuclear reactor was built in 1942 at the University of Chicago.
 A single-select problem: Is the question answered in a satisfactory fashion?

Select from the following.
 (1). yes
 (2). no
(2).
Fine-tuned:     a device that uses nuclear fission to produce electricity
Fine-tuned+RAG: a nuclear fuel

Retrieved context:
  [1] Nuclear reactors use fission reactions to vaporize water. The resulting steam is used to drive a turbine, which generates electricity....
  [2] Note Fission is the radioactive process used in nuclear power plants and one type of nuclear bomb....
  [3] Two nuclei must collide for fusion to occur. High temperatures are required to give the nuclei enough kinetic energy to overcome the very strong repul...


## 15. Improving the RAG Pipeline

The baseline RAG pipeline gets a lot of questions right, but there's room to improve. Two changes worth making: a better prompt template and hybrid search.

### Improvement 1: A sharper prompt

Our original prompt says "Use the following context to answer the question" — that's vague. The model sometimes drifts, adds information it remembers from pre-training, or generates more text than needed. A more directive prompt pins it down: answer from the context only, keep it short, don't speculate.

In [31]:
IMPROVED_PROMPT_TEMPLATE = """Answer the question below using ONLY the provided context.
Give a short, specific answer (a few words). Do not guess or add information beyond what's in the context.

Context:
{context}

Question: {question}
Answer:"""

print("Original prompt template:")
print(RAG_PROMPT_TEMPLATE)
print("\n" + "="*50)
print("\nImproved prompt template:")
print(IMPROVED_PROMPT_TEMPLATE)

Original prompt template:
Use the following context to answer the question. If the context doesn't contain enough information, say so.

Context:
{context}

Question: {question}
Answer:


Improved prompt template:
Answer the question below using ONLY the provided context.
Give a short, specific answer (a few words). Do not guess or add information beyond what's in the context.

Context:
{context}

Question: {question}
Answer:


### Improvement 2: Hybrid search (vector + keyword) with score normalization

So far we've only used vector similarity to find relevant passages. Vector search is good at matching meaning — a question about "rocks from volcanoes" will match a passage about "igneous formation" even though they share no words. But it can miss cases where an exact term matters, like the word "nucleus" or "luster" appearing literally in a passage.

**BM25** is a keyword-based scoring algorithm used by traditional search engines. It counts how often query words appear in a passage, weighted by how rare each word is across the whole corpus. Common words like "the" contribute almost nothing; specific terms like "nucleus" contribute a lot.

Combining both in a single query means a passage that matches semantically *and* contains the right keywords gets ranked higher.

**Score normalization:** BM25 and cosine similarity scores live on completely different scales — BM25 can range from 0 to 20+, while cosine similarity is 0 to 1. If you just add them together, BM25 dominates and vector similarity barely matters. We use **min-max normalization** here: rescale each set of scores to the 0–1 range independently, then average them. This makes the math visible and easy to follow.

**A note on Reciprocal Rank Fusion (RRF):** In production, many systems use RRF instead of score normalization. RRF throws away the raw scores entirely and only cares about the order each search method returns its results. Each result gets a score of 1 / (k + rank) from each method it appears in, where rank is its position (1st, 2nd, 3rd...) and k is a constant (typically 60) that controls how much rank differences matter. These per-method scores are then summed. For example, if keyword search returns Document A as its 1st result and semantic search returns it as its 3rd result, RRF assigns it a combined score of 1/(60+1) + 1/(60+3) = 0.0164 + 0.0159 = 0.0323. A document that appears 1st in both methods would score 1/(60+1) + 1/(60+1) = 0.0328 — higher, because both methods agree it's relevant. The large k value flattens the difference between adjacent ranks: being 1st vs 2nd in only one list barely matters compared to appearing in both lists at all. A smaller k would make rank position matter more; a larger k makes it matter even less. This makes RRF more robust than score normalization: a search method that gives one result an unusually high score won't dominate the final ranking. OpenSearch supports RRF natively through its search pipeline configuration. We use min-max normalization in this lab because it's more transparent and teaches the underlying concept, but if you're building a production hybrid search system, RRF is often the safer default.

In [20]:
def bm25_search(query, top_k=10):
    """Keyword-only search using BM25."""
    search_body = {
        "size": top_k,
        "query": {
            "match": {
                "text": query
            }
        }
    }
    response = os_client.search(index=INDEX_NAME, body=search_body)
    return {
        hit["_id"]: {"text": hit["_source"]["text"], "score": hit["_score"]}
        for hit in response["hits"]["hits"]
    }


def knn_search(query, top_k=10):
    """Vector-only search using cosine similarity."""
    query_embedding = embedder.encode(query).tolist()
    search_body = {
        "size": top_k,
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": top_k
                }
            }
        }
    }
    response = os_client.search(index=INDEX_NAME, body=search_body)
    return {
        hit["_id"]: {"text": hit["_source"]["text"], "score": hit["_score"]}
        for hit in response["hits"]["hits"]
    }


def min_max_normalize(scores):
    """Min-max normalization: maps all scores to the 0-1 range.
    This is the same technique OpenSearch's normalization-processor uses."""
    if not scores:
        return {}
    values = [s for s in scores.values()]
    min_val = min(values)
    max_val = max(values)
    if max_val == min_val:
        return {doc_id: 1.0 for doc_id in scores}
    return {
        doc_id: (score - min_val) / (max_val - min_val)
        for doc_id, score in scores.items()
    }


def hybrid_search(query, top_k=3):
    """Hybrid search mimicking OpenSearch's normalization-processor pipeline.
    
    Steps (same as OpenSearch's normalization-processor):
    1. Run knn and BM25 queries separately
    2. Min-max normalize each score set to 0-1
    3. Arithmetic mean of normalized scores
    4. Rank by combined score
    """
    # Step 1: Run both searches (fetch more than top_k so normalization has enough data)
    knn_results = knn_search(query, top_k=top_k * 3)
    bm25_results = bm25_search(query, top_k=top_k * 3)
    
    # Step 2: Min-max normalize each set independently
    knn_scores = {doc_id: r["score"] for doc_id, r in knn_results.items()}
    bm25_scores = {doc_id: r["score"] for doc_id, r in bm25_results.items()}
    
    norm_knn = min_max_normalize(knn_scores)
    norm_bm25 = min_max_normalize(bm25_scores)
    
    # Step 3: Combine with arithmetic mean
    all_doc_ids = set(norm_knn.keys()) | set(norm_bm25.keys())
    combined = {}
    for doc_id in all_doc_ids:
        knn_score = norm_knn.get(doc_id, 0.0)
        bm25_score = norm_bm25.get(doc_id, 0.0)
        combined[doc_id] = (knn_score + bm25_score) / 2
    
    # Step 4: Rank and return top_k
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    # Look up the actual text for each result
    all_results = {**knn_results, **bm25_results}
    results = []
    for doc_id, score in ranked:
        results.append({
            "text": all_results[doc_id]["text"],
            "score": score
        })
    
    return results


def answer_with_improved_rag(question, model, tokenizer, top_k=3):
    """RAG with hybrid search and improved prompt."""
    results = hybrid_search(question, top_k=top_k)
    context = "\n\n".join([r["text"] for r in results])
    prompt = IMPROVED_PROMPT_TEMPLATE.format(context=context, question=question)
    answer = generate_answer(model, tokenizer, prompt)
    return answer, results

### Baseline RAG vs Improved RAG

Same 10 questions, same model, same top_k=3. The only differences are the prompt template and the search strategy.

In [32]:
for i, q in enumerate(TEST_QUESTIONS):
    baseline_ans, _ = answer_with_rag(q, finetuned_model, tokenizer, top_k=3)
    improved_ans, _ = answer_with_improved_rag(q, finetuned_model, tokenizer, top_k=3)
    
    print(f"{'='*70}")
    print(f"Q{i+1}: {q}")
    print(f"  Expected:     {EXPECTED_ANSWERS[i]}")
    print(f"  Baseline RAG: {baseline_ans}")
    print(f"  Improved RAG: {improved_ans}")
    print()

Q1: What type of organism is commonly used in preparation of foods such as bread and yogurt?
  Expected:     Microorganisms (yeast, bacteria)
  Baseline RAG: bacteria
  Improved RAG: bacteria

Q2: What is the term for the property of a mineral that describes how it reflects light?
  Expected:     Luster
  Baseline RAG: luster
  Improved RAG: luster

Q3: What phenomenon makes global sea levels rise when ice sheets melt?
  Expected:     Thermal expansion of water
  Baseline RAG: melting glaciers
  Improved RAG: global warming

Q4: What is the main gas found in the air we breathe?
  Expected:     Nitrogen
  Baseline RAG: nitrogen
  Improved RAG: oxygen

Q5: What is the name of the process by which plants make their own food using sunlight?
  Expected:     Photosynthesis
  Baseline RAG: photosynthesis
  Improved RAG: photosynthesis

Q6: What are the building blocks of proteins called?
  Expected:     Amino acids
  Baseline RAG: amino acids
  Improved RAG: amino acids

Q7: What type of rock

## Wrapping up

Here's what we built across both notebooks:

1. **Lab 5a** — Took a general-purpose language model and fine-tuned it with LoRA on science Q&A. The model learned to produce clean, concise answers instead of noisy text completions. Beyond formatting, fine-tuning also adapts the model to a specific task and domain — it picked up science vocabulary and the Q&A pattern from the training data, making it more effective at this particular job than the generic base model.

2. **Lab 5b** — Added a retrieval layer using OpenSearch. The fine-tuned model gets relevant passages as context before answering, which corrects some of the knowledge gaps the model has on its own. We then explored improvements — a more directive prompt and hybrid search combining vector similarity with keyword matching.

**An important takeaway from the improvement section:** more sophisticated doesn't always mean better. In our case, the baseline RAG (vector-only search, simple prompt) actually outperformed the improved version on several questions. The hybrid search pulled in different passages that confused the model, and the stronger prompt didn't compensate. This is normal — RAG tuning is an iterative process. What works depends on the model size, the dataset, the questions, and how these pieces interact. There's no single configuration that wins everywhere, and the only way to find what works for your setup is to test, compare, and adjust.

Fine-tuning and RAG handle different parts of the problem. Fine-tuning shapes *how* the model responds and adapts it to a domain. RAG controls *what information* it has access to at query time. And within RAG itself, the search strategy, prompt design, and retrieval parameters all interact in ways that aren't always predictable — which is why evaluation matters.

### Things to think about

- What happens if the knowledge base doesn't contain anything relevant to the question? How does the model behave?
- How would you evaluate the quality of the retrieved passages? Not all high-scoring results are equally useful.
- In production, the knowledge base would be continuously updated. What does that mean for maintaining the vector index?
- Could you use RAG *without* fine-tuning? When would that be enough, and when wouldn't it?
- How would you handle score normalization between BM25 and vector similarity in a production system?